In [1]:
import pandas as pd

reduced_df = pd.read_feather('./processed/reduced_coerced_engagement_list.feather')

In [14]:
from datetime import datetime

# Search for FTTS Engagement Service Codes 10459 and 11420
codes = ['10459', '11420', '10170', '11420', '10466']
ftts = reduced_df[reduced_df['Engagement Service Code'].isin(codes)].copy()

for code in codes:
    df_code = reduced_df[reduced_df['Engagement Service Code'] == code].copy()
    count = len(df_code)
    total_etd = df_code['ANSR / Tech Revenue ETD'].sum()
    total_fytd = df_code.get('ANSR / Tech Revenue FYTD', pd.Series(dtype='float')).sum()
    print(f"Code {code}: {count:,} rows | ETD total: ${total_etd:,.2f} | FYTD total: ${total_fytd:,.2f}")
    # Save per-code review file
    # df_code.to_excel(f'./review/engagement_service_code_{code}.xlsx', index=False)

# Combined summary
combined_count = len(ftts)
combined_etd = ftts['ANSR / Tech Revenue ETD'].sum()
combined_fytd = ftts.get('ANSR / Tech Revenue FYTD', pd.Series(dtype='float')).sum()
print("Before filtering for Engagement 'FTTS':")
print(f"\nCombined codes {codes}: {combined_count:,} rows | ETD total: ${combined_etd:,.2f} | FYTD total: ${combined_fytd:,.2f}")
# Filter column Engagement for text 'FTTS' 
ftts = ftts[ftts['Engagement'].str.contains('FTTS', case=False, na=False)]
print("After filtering for Engagement 'FTTS':")
print(f"\nCombined codes {codes}: {len(ftts):,} rows | ETD total: ${ftts['ANSR / Tech Revenue ETD'].sum():,.2f} | FYTD total: ${ftts.get('ANSR / Tech Revenue FYTD', pd.Series(dtype='float')).sum():,.2f}")
# Save the file with a data/time stamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ftts.to_excel(f'./review/ftts_only_{timestamp}.xlsx', index=False)

Code 10459: 1,736 rows | ETD total: $295,184,045.14 | FYTD total: $32,649,304.69
Code 11420: 38,434 rows | ETD total: $780,749,970.76 | FYTD total: $129,092,955.19
Code 10170: 0 rows | ETD total: $0.00 | FYTD total: $0.00
Code 11420: 38,434 rows | ETD total: $780,749,970.76 | FYTD total: $129,092,955.19
Code 10466: 1,045 rows | ETD total: $30,800,185.42 | FYTD total: $3,886,836.65
Before filtering for Engagement 'FTTS':

Combined codes ['10459', '11420', '10170', '11420', '10466']: 41,215 rows | ETD total: $1,106,734,201.32 | FYTD total: $165,629,096.53
After filtering for Engagement 'FTTS':

Combined codes ['10459', '11420', '10170', '11420', '10466']: 1,970 rows | ETD total: $298,909,147.00 | FYTD total: $33,546,916.23


In [3]:
# Import engagement list from ShareTrust
sharetrust_engagements = pd.read_excel('./raw/ftts/FTTS FY25 EYST Adoption Analysis 07182025.xlsx', sheet_name='Engagement Detail FY25_6.27.25')


In [4]:
# Perform the equivalent of an XLookup to match engagements in sharetrust_engagements with ftts DataFrame
# Client ID in ftts to match with Client ID in sharetrust_engagements
# Only bring the columns 'Roll Up Client', 'EYST Active Client (client roll up version)', and 'EYST Data Clean-Up Active Client (client roll up version)'
# If the merged value is missing, fill with 'No'
ftts_merged = ftts.merge(
    sharetrust_engagements[['Client ID', 'Roll Up Client', 'EYST Active Client (client roll up version)', 'EYST Data Clean-Up Active Client (client roll up version)']],
    how='left',
    left_on='Client ID',
    right_on='Client ID'
)

# Fill missing values in the new columns with 'N/A'
ftts_merged['Roll Up Client'].fillna('N/A', inplace=True)
ftts_merged['EYST Active Client (client roll up version)'].fillna('N/A', inplace=True)
ftts_merged['EYST Data Clean-Up Active Client (client roll up version)'].fillna('N/A', inplace=True)

# De-duplicate rows - keep first occurrence of each unique row
print(f"Rows before deduplication: {len(ftts_merged):,}")
ftts_merged = ftts_merged.drop_duplicates()
print(f"Rows after deduplication: {len(ftts_merged):,}")

ftts_merged.to_excel(f'./review/ftts_combined_with_sharetrust_{timestamp}.xlsx', index=False)

/var/folders/v4/p14n02tj5ysf5n_ccn8pq6g00000gn/T/ipykernel_98002/2977328654.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  ftts_merged['Roll Up Client'].fillna('N/A', inplace=True)
/var/folders/v4/p14n02tj5ysf5n_ccn8pq6g00000gn/T/ipykernel_98002/2977328654.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting 

Rows before deduplication: 14,597
Rows after deduplication: 1,978


In [13]:
# Calculate requested metrics using ftts_merged
import numpy as np

CLEANUP_COL = 'EYST Data Clean-Up Active Client (client roll up version)'
REVENUE_COL = 'ANSR / Tech Revenue ETD'
CLIENT_ID_COL = 'Client ID'
STATUS_COL = 'Engagement Status'

# Safety checks
required_cols = {CLEANUP_COL, REVENUE_COL, CLIENT_ID_COL, STATUS_COL}
missing = [c for c in required_cols if c not in ftts_merged.columns]
if missing:
    raise KeyError(f"Missing required columns in ftts_merged: {missing}")

# Filter out rows with 'N/A' cleanup and 'Cancelled' status
ftts_filtered = ftts_merged[(ftts_merged[CLEANUP_COL] != 'N/A') & (ftts_merged[STATUS_COL] != 'Cancelled')].copy()

# Normalize the cleanup flag to detect 'yes' values case-insensitively
cleanup_yes_mask = (
    ftts_filtered[CLEANUP_COL]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq('yes')
)

# Metric 1: Revenue $ = Total ETD ANSR / ETD ANSR where cleanup flag is 'yes'
total_etd = ftts_filtered[REVENUE_COL].fillna(0).sum()
yes_etd = ftts_filtered.loc[cleanup_yes_mask, REVENUE_COL].fillna(0).sum()
revenue_metric = (total_etd / yes_etd) if yes_etd else np.nan

# Distinct client counts (unique clients only)
total_clients = ftts_filtered[CLIENT_ID_COL].nunique()
yes_clients = ftts_filtered.loc[cleanup_yes_mask, CLIENT_ID_COL].nunique()

# Metric 2: % Engagements = Yes clients / Total clients (as a percent)
percent_engagements = (yes_clients / total_clients * 100) if total_clients else np.nan

# Metric 3: Client Ratio = Yes clients / Total clients (ratio under 1.0)
client_ratio = (yes_clients / total_clients) if total_clients else np.nan

print('Metrics (computed from ftts_merged, excluding N/A and Cancelled):')
print(f"Revenue $ (Total ETD / Yes ETD): {revenue_metric:,.4f} | Total ETD: ${total_etd/1e6:,.1f}M | Yes ETD: ${yes_etd/1e6:,.1f}M")
print(f"% Engagements (Yes clients / Total clients): {percent_engagements:,.2f}% | Yes clients: {yes_clients:,} | Total clients: {total_clients:,}")
print(f"Client Ratio (Yes clients / Total clients): {client_ratio:,.4f}")

Metrics (computed from ftts_merged, excluding N/A and Cancelled):
Revenue $ (Total ETD / Yes ETD): 1.2571 | Total ETD: $265.0M | Yes ETD: $210.8M
% Engagements (Yes clients / Total clients): 61.17% | Yes clients: 178 | Total clients: 291
Client Ratio (Yes clients / Total clients): 0.6117
